# SPI and SPEI from a prepared Zarr store with xarray and Dask

This notebook walks the end-to-end workflow behind the `climate_indices`
package's tutorial example:

```text
prepared Zarr  ->  inspect and validate (xarray)  ->  compute SPI/SPEI (Dask)
     ->  write Zarr  ->  reopen  ->  maps and time series
```

**Scientific question.** How dry or wet was each month of the 1980-2016 record
relative to the 1981-2010 Calibration Period, for every cell of the example
nClimGrid grid?

**Expected outputs.** A `climate_indices_output.zarr` store holding a 3-month
SPI and a 3-month SPEI per grid cell and month, then spatial maps and a time
series plot read back from that store.

**Learning objectives.** After working through this notebook you will be able
to:

1. Explain the distinct roles of Zarr (storage), xarray (labeled model), and
   Dask (lazy scheduling) in this workflow.
2. Open the prepared store lazily and inspect its variables, coordinates,
   units, masks, and chunks.
3. State and check the input contract that SPI/SPEI correctness depends on.
4. Run the public `climate_indices.spi` and `climate_indices.spei` xarray APIs
   over a Dask-backed dataset and persist the result.
5. Select a calendar date from the reopened output and compare labeled SPI
   and SPEI maps while materializing only their 2-D slices.

The results here are illustrative: they demonstrate the package's xarray/Dask
path on a reduced sample dataset. They are not a validated drought assessment
of any location.

## Terms used throughout

- **SPI (Standardized Precipitation Index)**: precipitation deficit or surplus
  standardized against the distribution of precipitation at each location and
  calendar month.
- **SPEI (Standardized Precipitation-Evapotranspiration Index)**: the same
  standardization applied to the water balance (precipitation minus potential
  evapotranspiration), so it accounts for atmospheric demand.
- **PET (potential evapotranspiration)**: the water that would evaporate and
  transpire under the given atmospheric conditions, as monthly totals here.
- **Timescale**: the accumulation window, `scale` in code. SPI/SPEI at a
  3-month Timescale compare overlapping 3-month totals; at 12 months they
  compare annual water availability.
- **Periodicity**: the observation frequency, monthly here. Periodicity is
  distinct from Timescale: monthly observations can feed any Timescale.
- **Calibration Period**: the years used to fit the reference distribution
  (1981-2010 in this example). It must cover a complete record for every
  location, otherwise fitted distributions are not comparable across cells.

The indices are dimensionless standardized anomalies, not water amounts.
Negative values mean drier than the Calibration Period, positive means wetter,
and values near zero are near normal.

## Environment and data setup

From the repository root:

```bash
uv sync --group dev
uv run jupyter lab notebooks/zarr_dask_spi_spei.ipynb
```

The `dev` dependency group provides Jupyter, Dask's `distributed`, matplotlib,
and Zarr. The kernel's working directory is this notebook's directory
(`notebooks/`), which is why the default data path below is `../data/e2e`.
This teaching notebook lives in `notebooks/`; the one-time input preparation
support lives in `scripts/`.

### One-time input preparation (optional)

The prepared store is the normal tutorial entry point. To regenerate it:

```bash
uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py
```

- Downloads about 5 MB of source NetCDF, verified against pinned SHA-256
  digests; reruns use the cache under `data/e2e/source/` and need no network
  access.
- Validates the result, then atomically switches `data/e2e/current` to the new
  generation.
- `data/e2e/` is gitignored; never commit it.

### Source attribution

The example inputs are a reduced and modified sample of the NOAA/NCEI
nClimGrid v1 dataset, pinned at `monocongo/example_climate_indices` commit
`ae57c488af832c1ebfdf864c8ed7d16636e2e36f`. Precipitation and PET are monthly
totals over a subset grid and period; the units declared in the source are
relabeled to `mm` only after they are verified, never numerically converted.

- Vose, Russell S., Applequist, Scott, Squires, Mike, Durre, Imke, Menne,
  Matthew J., Williams, Claude N. Jr., Fenimore, Chris, Gleason, Karin, and
  Arndt, Derek (2014): NOAA Monthly U.S. Climate Gridded Dataset (NClimGrid),
  Version 1 (reduced subset). NOAA National Centers for Environmental
  Information. DOI:10.7289/V5SX6B56
- No NOAA endorsement is implied, and this sample is not the unaltered
  dataset.
- Background on the upstream provenance and redistribution constraints
  this sample inherits:
  `docs/research/nclimgrid-acquisition-and-redistribution.md`.

## Configuration and data location

One configuration literal drives everything below. The
`CLIMATE_INDICES_E2E_DATA` environment variable overrides the data directory
(the test suite uses this); otherwise the default `../data/e2e` resolves
relative to this notebook.

In [ ]:
import json
import os
import shutil
import uuid
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from climate_indices import compute, indices, spei, spi
from climate_indices.exceptions import CoordinateValidationError, InvalidArgumentError

# Canonical configuration: matches data/e2e/manifest.json (1980-2016 monthly
# inputs, 1981-2010 Calibration Period).
pipeline_config = {
    "scale": 3,  # 3-Month SPI/SPEI
    "distribution_spi": indices.Distribution.gamma,
    "distribution_spei": indices.Distribution.pearson,
    "data_start_year": 1980,
    "cal_start_year": 1981,
    "cal_end_year": 2010,
    "periodicity": compute.Periodicity.monthly,
}

# scripts/prepare_e2e_inputs.py atomically publishes the generation that
# data/e2e/current points at.
data_root = Path(os.environ.get("CLIMATE_INDICES_E2E_DATA", "../data/e2e"))
if not (data_root / "current").exists():
    raise FileNotFoundError(
        f"Prepared inputs not found: {data_root / 'current'}. "
        "Generate them once with: uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py"
    )
data_dir = (data_root / "current").resolve()
prepared_zarr = data_dir / "cache_prepared_input.zarr"
final_output_zarr = (data_root / "climate_indices_output.zarr").resolve()
# Results go to a separate store staged beside its final name, so a failed run
# never truncates the completed output and a rerun never reads a half-written one.
staging_path = final_output_zarr.with_name(f".{final_output_zarr.name}.staging-{uuid.uuid4().hex}")

# Reject overlapping stores before anything is opened, written, or deleted:
# writing into the prepared inputs (or the reverse) would destroy them on the
# next rerun, and nesting the staging directory inside either path would mix
# partial and complete data.
for first, second in combinations((prepared_zarr.resolve(), final_output_zarr, staging_path), 2):
    if first == second or first in second.parents or second in first.parents:
        raise InvalidArgumentError(
            "Prepared inputs, output store, and staging path must not overlap.",
            argument_name="CLIMATE_INDICES_E2E_DATA",
            argument_value=f"{first} and {second}",
            valid_values="separate, non-nested paths",
        )


In [ ]:
manifest_path = data_dir / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Input manifest not found: {manifest_path}. "
        "Regenerate the inputs with: uv run --with h5py --with zarr scripts/prepare_e2e_inputs.py"
    )
manifest = json.loads(manifest_path.read_text())
manifest

## Three complementary layers: Zarr, xarray, Dask

- **Zarr** is chunked array storage on disk. `cache_prepared_input.zarr` holds
  the numbers; on its own it does not know what "lat" or a month means beyond
  array shape.
- **xarray** is the labeled model on top. A `Dataset` is a container of named
  variables that share coordinates; a `DataArray` is one variable with its
  coordinates and attributes; a plain NumPy array carries values and shape
  only. Labels are what let `xr.align`, `.sel`, and the index functions match
  precipitation to PET by coordinate instead of by position.
- **Dask** is lazy scheduling. An operation on a Dask-backed `DataArray` only
  builds a task graph; the tasks run when you call `.compute()`, `.load()`, or
  write with `.to_zarr()`.

The roles are complementary, not interchangeable: Zarr without xarray loses
labels and metadata, xarray without Dask would materialize the whole grid, and
Dask without either would schedule unlabeled blocks.

## Open the prepared store lazily

`xr.open_zarr(..., consolidated=True)` reads the store's consolidated metadata
and returns a `Dataset` whose variables are Dask arrays: nothing is loaded into
memory yet. The store's own layout is one chunk along `time` — required so
distribution fitting sees each full time series — with 10x10 spatial blocks
written during preparation. Spatial chunking and worker settings are covered
in the Dask execution section below; here we only inspect what already
exists.

In [ ]:
ds = xr.open_zarr(prepared_zarr, consolidated=True)
ds

In [ ]:
ds["precip"]

In [ ]:
print("dims:", ds["precip"].dims)
print("time:", str(ds.time.values[0])[:10], "to", str(ds.time.values[-1])[:10], f"({ds.sizes['time']} monthly steps)")
print("lat:", float(ds.lat.min()), "to", float(ds.lat.max()), f"({ds.sizes['lat']} cells)")
print("lon:", float(ds.lon.min()), "to", float(ds.lon.max()), f"({ds.sizes['lon']} cells)")
print("dtype/units:", ds["precip"].dtype, ds["precip"].attrs.get("units"))
print("existing chunks:", ds["precip"].chunks)

# A reduction materializes only what it needs, not the whole grid.
print(f"Grid-mean monthly precip: {float(ds['precip'].mean('time').compute().mean()):.1f} mm")

## Input contract: required for correctness

Everything the calculation needs from its inputs, and the failure each check
rules out. These are correctness requirements, not performance preferences.

In [ ]:
def _validate_monthly_time(time_values):
    """Return complete month-start or month-end timestamps."""
    try:
        time = pd.DatetimeIndex(time_values)
    except (TypeError, ValueError, OverflowError) as exc:
        raise CoordinateValidationError(
            "Time coordinate must contain supported datetime values.",
            coordinate_name="time",
            reason="not datetime-like",
        ) from exc
    if time.empty:
        raise CoordinateValidationError(
            "Time coordinate must not be empty.", coordinate_name="time", reason="empty coordinate"
        )
    if time.is_month_start.all():
        expected_time = pd.date_range(time[0], periods=time.size, freq="MS")
    elif time.is_month_end.all():
        expected_time = pd.date_range(time[0], periods=time.size, freq=pd.offsets.MonthEnd())
    else:
        expected_time = pd.DatetimeIndex([])
    if not time.equals(expected_time):
        raise CoordinateValidationError(
            "Time coordinate must be a complete, chronological sequence of monthly "
            "month-start or month-end timestamps.",
            coordinate_name="time",
            reason="non-monotonic, irregular, or gapped monthly timestamps",
        )
    return time

**Time axis.** Complete, chronological monthly steps with whole calendar
years, starting in `data_start_year`, and a Calibration Period inside the data
range. A gap or a partial year shifts every later accumulation window and
silently misaligns the calibration fit.

In [ ]:
time = _validate_monthly_time(ds["time"].values)
if time[0].month != 1 or time[-1].month != 12:
    raise CoordinateValidationError(
        "Prepared dataset must cover complete calendar years.",
        coordinate_name="time",
        reason="incomplete first or final year",
    )
data_start_year = pipeline_config["data_start_year"]
if time[0].year != data_start_year:
    raise InvalidArgumentError(
        "pipeline_config['data_start_year'] does not match the prepared dataset's first monthly timestamp.",
        argument_name="data_start_year",
        argument_value=str(data_start_year),
        valid_values=str(time[0].year),
    )
cal_start_year, cal_end_year = pipeline_config["cal_start_year"], pipeline_config["cal_end_year"]
if not (data_start_year <= cal_start_year <= cal_end_year <= time[-1].year):
    raise InvalidArgumentError(
        "Calibration Period must fall within the prepared dataset's covered years.",
        argument_name="cal_start_year/cal_end_year",
        argument_value=f"{cal_start_year}-{cal_end_year}",
        valid_values=f"{data_start_year}-{time[-1].year}",
    )

calibration_observations = int(((time.year >= cal_start_year) & (time.year <= cal_end_year)).sum())
print(f"Calendar years: {time[0].year}-{time[-1].year} ({time.size} monthly steps)")
print(f"Calibration Period: {cal_start_year}-{cal_end_year}, {calibration_observations} monthly observations per cell")

**Coordinates.** Precipitation and PET must be identical before they are
combined. xarray's default behavior for binary operations is to *intersect*
coordinates, so a mismatched axis would silently shrink the result; the
preparation step therefore aligned them with `join="exact"` and SPEI is never
allowed to lean on an inner join.

In [ ]:
precip_exact, pet_exact = xr.align(ds["precip"], ds["pet"], join="exact")
print(f"precip and pet aligned exactly: {precip_exact.sizes} / {pet_exact.sizes}")

**Units.** Both variables must be monthly totals in millimeters, declared as
`units == "mm"`. Relabeling an attribute is not a conversion: a source in
inches or in daily rates would produce wrong index values while looking
perfectly labeled.

In [ ]:
for name in ("precip", "pet"):
    units = ds[name].attrs.get("units")
    if units != "mm":
        raise InvalidArgumentError(
            "Prepared inputs must be monthly totals with units='mm'; relabeling an attribute is not a conversion.",
            argument_name=f"{name}.units",
            argument_value=repr(units),
            valid_values="mm",
        )
    print(f"{name}: units={units!r}, dtype={ds[name].dtype}")

**Masks and zeros.** Missing values are `NaN`, meaningful zeros are data.
Precipitation and PET missingness must match at every timestamp, or SPEI
combines present PET with missing precipitation. Cells that are entirely `NaN`
carry no data at all; partially missing cells would bias the fitted
distribution and are rejected.

In [ ]:
precip, pet = ds["precip"], ds["pet"]
if not precip.isnull().equals(pet.isnull()):
    raise InvalidArgumentError(
        "Precipitation and PET missingness must match at every timestamp; "
        "SPEI would otherwise combine present values with missing ones.",
        argument_name="pet",
        argument_value="missingness differs from precip",
        valid_values="identical precip/pet missingness",
    )

all_missing = precip.isnull().all("time").load()
partially_missing = precip.isnull().any("time") & ~all_missing
if partially_missing.any():
    raise InvalidArgumentError(
        "Prepared inputs must not contain partially missing cells; the fitted distribution "
        "needs a complete series per cell.",
        argument_name="precip",
        argument_value=f"{int(partially_missing.sum())} partially missing cells",
        valid_values="complete series or all-missing cells",
    )
print(f"All-missing cells: {int(all_missing.sum())} of {all_missing.size}")
print(f"Partially missing cells: {int(partially_missing.sum())}")
print(f"Zero precipitation observations: {int((precip == 0).sum())} (valid data, not missing)")
print(f"Cells with any missing month: {int(precip.isnull().any('time').sum())}")

**Timescale gap, time chunking, and manifest agreement.** A Timescale of *n*
months cannot produce a value until *n − 1* accumulation months exist, so the
first *n − 1* values per cell are unavailable. On disk, `time` must remain a
single chunk so each cell's full series reaches the distribution fit
([ADR-0003](../docs/adr/0003-dask-time-dimension-single-chunk.md)). Finally,
the store must agree with the manifest that describes it.

In [ ]:
leading_unavailable = pipeline_config["scale"] - 1
print(f"Timescale {pipeline_config['scale']} months: first {leading_unavailable} values unavailable per cell")

time_chunks = ds["precip"].chunks[0]
print(f"Time chunks: {time_chunks}")
if len(time_chunks) != 1:
    raise CoordinateValidationError(
        "Prepared store must keep time as a single chunk.",
        coordinate_name="time",
        reason="multiple time chunks",
    )

store_chunks = list(ds["precip"].encoding["chunks"])
if dict(ds.sizes) != manifest["dimensions"] or store_chunks != list(manifest["chunks"]):
    raise InvalidArgumentError(
        "Prepared store dimensions or chunks disagree with manifest.json.",
        argument_name="manifest",
        argument_value=f"dims={dict(ds.sizes)}, chunks={store_chunks}",
        valid_values=f"dims={manifest['dimensions']}, chunks={manifest['chunks']}",
    )
print(f"Manifest agrees with the store: {manifest['dimensions']}, chunks {manifest['chunks']}")

### Chunks: storage layout versus execution layout

Zarr keeps each array in fixed-size on-disk chunks. The prepared store uses one
full `time` chunk and 10x10 `lat`/`lon` blocks, as the manifest check above
confirmed. `xr.open_zarr` exposes those storage chunks as the initial Dask
chunks, which is why `ds["precip"].chunks` already shows a layout before
anything is rechunked.

Dask execution chunks decide how the scheduler splits the work, and they can
change without rewriting the store: `.chunk({"time": -1})` in the calculation
below is a metadata operation. Each location's full series has to reach the
distribution fit as one piece, so `time` stays a single chunk, while the
spatial blocks are independent and become separate tasks. On this 38 x 87 grid
there are 4 `lat` blocks by 9 `lon` blocks, so the canonical calculation
exposes 36 independent spatial tasks per variable. That is the parallelism Dask
can actually use, whatever a client's worker count happens to be.


**The full-`time`-in-one-chunk rule is correctness, not tuning.** A split
`time` gives each chunk only part of a location's history, so a distribution
fit would run on statistics from a truncated series — silently different
numbers rather than a slower answer. The adapter rejects the call instead. The
error message carries the fix (`data = data.chunk({'time': -1})`), and the
code after the calculation cell triggers the failure deliberately, then
applies that fix.

Spatial chunk sizes, worker counts, and memory limits are the tuning knobs
here; they change speed and memory use, not results.


### An optional local client

The default threaded scheduler can run a sample this size. A local `Client` is
shown here so its settings and the diagnostic dashboard are visible before they
matter at larger scale. The client below runs four worker processes with one
thread each: separate processes dodge the GIL and keep each task independent.
`memory_limit` is enforced per worker, so a fixed value multiplies across the
workers; each limit here is derived from this host's physical memory (about
half, split four ways) and acts as a spill safety valve rather than a target.
`bokeh` is optional; when it is installed the client prints a dashboard URL,
where the **Task Stream** shows spatial blocks starting and finishing and
**Workers** shows memory. Without `bokeh` the client still runs, just headless.

A lower-resource alternative keeps one process and uses threads, which share
memory with no inter-process copies but contend for the GIL; the commented cell
after the client shows it. Either way, close the client when finished so worker
processes and the dashboard do not outlive the run.


In [ ]:
import psutil
from dask.distributed import Client

# Local client: four worker processes with one thread each, so GIL-bound
# per-block work can run in parallel. With bokeh installed, `client` prints a
# dashboard URL; the Task Stream there shows one row per worker as the spatial
# blocks run. Without bokeh it still works, headless.
# `memory_limit` is per worker, so a fixed value multiplies across the workers;
# deriving each limit from this host's physical memory (about half, split four
# ways) keeps the aggregate below the machine's capacity. The limit is a spill
# safety valve, not a target.
worker_memory_gb = max(1, int(psutil.virtual_memory().total * 0.5) // (4 * 2**30))
client = Client(n_workers=4, threads_per_worker=1, memory_limit=f"{worker_memory_gb}GB")
client


In [ ]:
# Lower-resource alternative: one process, two threads, a smaller memory limit.
# Threads share memory (no inter-process copies), but the GIL serializes
# Python-level work.
# client = Client(processes=False, n_workers=1, threads_per_worker=2, memory_limit="2GB")


## Canonical calculation path

SPI and SPEI are computed through the public typed API — `climate_indices.spi` and
`climate_indices.spei` — on Dask-backed `xarray.DataArray`s. Under the hood the adapter
runs `xr.apply_ufunc(..., dask="parallelized")`, so labeled dimensions and coordinates
are preserved and results stay lazy until the deliberate Zarr write below
([ADR-0001](../docs/adr/0001-dual-numpy-xarray-api.md),
[ADR-0002](../docs/adr/0002-multiprocessing-cli-dask-xarray.md)).

Dask parallelizes across spatial chunks: each `lat`/`lon` block of the prepared store is
an independent task, while `time` stays a single chunk so distribution fitting sees the
full series at each location
([ADR-0003](../docs/adr/0003-dask-time-dimension-single-chunk.md)). Worker count alone
is not evidence of parallelism — inspect the chunk layout and task graph instead.
Precipitation and PET were aligned with `join="exact"` at preparation time, so SPEI
never relies on xarray's coordinate intersection.

In [ ]:
ds_calc = ds.transpose("time", "lat", "lon").chunk({"time": -1})

index_kwargs = {
    "scale": pipeline_config["scale"],
    "data_start_year": data_start_year,
    "calibration_year_initial": cal_start_year,
    "calibration_year_final": cal_end_year,
    "periodicity": pipeline_config["periodicity"],
}
spi_da = spi(values=ds_calc["precip"], distribution=pipeline_config["distribution_spi"], **index_kwargs)
spei_da = spei(
    precips_mm=ds_calc["precip"], pet_mm=ds_calc["pet"], distribution=pipeline_config["distribution_spei"], **index_kwargs
)
spi_name, spei_name = f"spi_{pipeline_config['scale']}", f"spei_{pipeline_config['scale']}"
ds_output = xr.Dataset({spi_name: spi_da, spei_name: spei_da}, coords=ds_calc.coords)

# The public API attaches index identity and provenance to each result. The CF
# registry deliberately provides no standard_name for SPI or SPEI, so any
# inherited input name (here precipitation's precipitation_amount) is dropped
# rather than mislabeling a drought index; periodicity records that scale units
# are months, not steps of an unspecified length.
for variable_name in (spi_name, spei_name):
    ds_output[variable_name].attrs.pop("standard_name", None)
    ds_output[variable_name].attrs["periodicity"] = pipeline_config["periodicity"].name


In [ ]:
# Storage chunks are the Zarr layout; execution chunks are how Dask partitions
# the same arrays. Both are metadata, so nothing is read here.
print("storage chunks (time, lat, lon):", ds["precip"].encoding["chunks"])
print("execution blocks (time, lat, lon):", tuple(len(chunks) for chunks in ds_calc["precip"].chunks))

# The task graph is also just a data structure until a scheduler runs it.
# Summing each layer's compact length counts tasks without flattening the
# graph into one key per task, so the diagnostic stays cheap as the grid grows;
# the graph inspected here is the one the write below will execute.
for name, variable in ((spi_name, spi_da), (spei_name, spei_da)):
    data = variable.data
    lat_blocks, lon_blocks = len(data.chunks[-2]), len(data.chunks[-1])
    task_graph = data.dask
    task_count = sum(len(layer) for layer in task_graph.layers.values())
    print(
        f"{name}: {lat_blocks} x {lon_blocks} = {data.npartitions} spatial tasks, "
        f"{len(task_graph.layers)} graph layers, {task_count} tasks total"
    )


In [ ]:
# The guard fires synchronously, before anything is scheduled, and the message
# names the fix. Split `time` on purpose to see it, then rechunk it back.
mis_chunked = ds_calc["precip"].chunk({"time": 12})
try:
    spi(values=mis_chunked, distribution=pipeline_config["distribution_spi"], **index_kwargs)
except CoordinateValidationError as error:
    print(f"rejected as expected: {error}")

rechunked = mis_chunked.chunk({"time": -1})
print(
    "after data.chunk({'time': -1}): "
    f"{tuple(len(chunks) for chunks in rechunked.chunks)} execution blocks (time, lat, lon)"
)


### Running the graph: `.compute()`, `.persist()`, and `.to_zarr()`

Everything built so far is a lazy graph: `ds_calc`, `spi_da`, `spei_da`, and
`ds_output` hold no index values yet, and the earlier grid-mean `.compute()` was
a separate small reduction. The ways to run the graph differ in where the
result ends up:

- `.compute()` executes and returns local NumPy arrays.
- `.persist()` executes and keeps the result in the cluster's memory as Dask
  arrays, so later steps reuse it without recomputing. That memory stays
  occupied until the objects are released or the client closes.
- `.to_zarr()` streams each completed block to disk, materializing `spi_da` and
  `spei_da` once without collecting the full grid in the client process first.

The write below is the only full materialization of the index grids in this
notebook; `.compute()` and `.persist()` are not invoked on the full index
arrays just to show them. Later
cells compute small selections from the reopened store, which is the pattern
for inspection; the round-trip check re-derives one cell from the in-memory
`ds_output` graph.

Chunk size and worker settings trade off against each other. Many small chunks
give the scheduler more independent tasks but add per-task overhead and more
graph bookkeeping; fewer large chunks cut that overhead but limit parallelism
and raise peak memory per task. Worker processes avoid the GIL at the cost of
process startup and IPC; threads share memory but contend for the GIL.
`memory_limit` is a per-worker spill threshold, not a target. This 36-block
sample is small enough that Dask is optional — no speedup is promised here, and
the point is the mechanism, not a benchmark.


## Persist the results

The write below creates a separate store, `data/e2e/climate_indices_output.zarr`, so the prepared
inputs stay read-only and a rerun never reads the output it is replacing. It is Zarr **v2** written
with `consolidated=True`: one `.zmetadata` document holds all store metadata, so readers open the
store with a single metadata read. Index variables are stored as `float32`, the precision the
`float32` mm inputs carry, even though the typed API computes in `float64`; downcasting at write
time keeps the on-disk precision honest.

Each variable keeps the API's `long_name`, dimensionless `units`, `references`, `scale`,
`distribution`, Calibration Period years, `climate_indices_version`, and `history`, plus the
`periodicity` attribute added above that makes the Timescale unit explicit. The API provides no
`standard_name` for these indices because CF defines none.

**Reruns and replacement.** A run stages the complete store beside the target (a unique
`.staging-<uuid>` sibling, collision-free for concurrent runs) and then swaps directories: the
previous output is renamed aside and only removed after the new store is in place. A failed staging
write is removed while the previous completed output stays intact. Replacement is a
local, single-writer operation — close readers before rerunning, and do not point concurrent
writers at the same output path; the final directory swap is not crash-atomic. The path check in
the configuration cell rejects overlapping input, output, and staging paths before anything is
opened or deleted.


In [ ]:
# Stage under the private name chosen and checked above, then swap by rename:
# renaming the previous generation aside (fast, atomic) instead of deleting it
# first means an interruption mid-publish leaves the previous completed
# generation recoverable instead of losing it outright.
# The typed API computes in float64 (xr.apply_ufunc(..., output_dtypes=[float])
# regardless of input dtype); downcast on write only, to keep the on-disk
# footprint at the float32 precision the float32 mm inputs actually carry.
float32_encoding = {"dtype": "float32"}
# Persist annual time chunks so a one-date read touches one chunk per spatial
# block instead of decoding the full record; the fitting input still needs
# time as a single chunk (ADR-0003).
ds_output = ds_output.chunk({"time": 12})
try:
    ds_output.to_zarr(
        staging_path,
        mode="w",
        zarr_format=2,
        consolidated=True,
        encoding={spi_name: float32_encoding, spei_name: float32_encoding},
    )
    backup_path = final_output_zarr.with_name(f".{final_output_zarr.name}.previous")
    shutil.rmtree(backup_path, ignore_errors=True)
    if final_output_zarr.exists():
        final_output_zarr.rename(backup_path)
    try:
        staging_path.rename(final_output_zarr)
    except BaseException:
        if backup_path.exists():
            backup_path.rename(final_output_zarr)
        raise
    shutil.rmtree(backup_path, ignore_errors=True)
finally:
    shutil.rmtree(staging_path, ignore_errors=True)
    # Close the local client on the success and failure paths alike, so a
    # failed write never leaves worker processes and the dashboard running.
    # The guard supports running this cell without a client; the reopened
    # store below is read with the default local scheduler.
    if "client" in globals():
        client.close()


## Reopen the saved results

`xr.open_zarr(final_output_zarr, consolidated=True)` opens a new Dataset handle from the persisted
store. Its variables are lazy Dask arrays, so inspecting metadata and chunks below reads no values.
Persisting the indices is what makes this reopening possible later, independently of the prepared
inputs — the diagnostics and plots that follow select from this handle and compute only what they
need, then the handle is closed. Nothing in this section recomputes the full
index grids; only the round-trip check re-derives one location's series from
the in-memory `ds_output` graph.


In [ ]:
# A fresh handle opened from disk: these variables are Dask arrays backed by
# the saved store, not the in-memory ds_output used to write it.
out_ds = xr.open_zarr(final_output_zarr, consolidated=True)


In [ ]:
out_ds

In [ ]:
# Metadata and chunk inspection reads no data: only consolidated metadata is
# loaded, and every variable is still a lazy Dask array.
for variable_name in (spi_name, spei_name):
    variable = out_ds[variable_name]
    print(f"{variable_name}: dtype={variable.dtype}, chunks={variable.chunks}")
    for key in (
        "long_name",
        "units",
        "scale",
        "periodicity",
        "distribution",
        "calibration_year_initial",
        "calibration_year_final",
        "climate_indices_version",
    ):
        print(f"  {key}: {variable.attrs.get(key)!r}")


In [ ]:
# Round-trip check: the reopened store must reproduce the calculated values at
# float32 storage precision and the metadata the API attached. Only one
# location's series is materialized here; the full grids stay on disk.

for variable_name in (spi_name, spei_name):
    stored, calculated = out_ds[variable_name], ds_output[variable_name]
    np.testing.assert_allclose(
        stored.isel(lat=0, lon=0).values,
        calculated.isel(lat=0, lon=0).values,
        rtol=1e-5,
        atol=1e-5,
        equal_nan=True,
    )
    for key, expected in calculated.attrs.items():
        assert stored.attrs.get(key) == expected, f"{variable_name}.{key}: {stored.attrs.get(key)!r} != {expected!r}"
    assert "standard_name" not in stored.attrs
np.testing.assert_array_equal(out_ds.time.values, ds.time.values)
print(f"Round-trip check passed for {spi_name} and {spei_name}")


## Maps from the reopened output

The maps below read from `out_ds`, the fresh handle backed by the saved output store, not the
in-memory `ds_output` calculation result. Set `map_date` to any exact calendar date shown in
`out_ds.time`. `.sel(time=map_date)` selects by coordinate label; `.isel(time=9)` would instead
select the tenth stored position and could silently change dates if the record changed. Selection
happens before `.compute()`, so the result contains only the two selected 2-D slices. The output
store is written with annual time chunks, so a one-date read touches one time chunk per spatial
block rather than decoding the full record. (The single time chunk required for distribution
fitting, [ADR-0003](../docs/adr/0003-dask-time-dimension-single-chunk.md), applies to the
calculation input, not the persisted store.) Other months remain lazy in `out_ds`. An unavailable
date raises `KeyError` rather than silently choosing a neighbor.


In [ ]:
# Accepted stores use month-start or month-end timestamps, so the default
# comes from the reopened store rather than a hard-coded label; choose any
# position shown in out_ds.time.
map_date = pd.Timestamp(out_ds.time.values[9]).strftime("%Y-%m-%d")
map_slices = out_ds[[spi_name, spei_name]].sel(time=map_date).compute()
selected_date = pd.Timestamp(map_slices.time.item()).strftime("%Y-%m-%d")
map_slices

Both maps use the same continuous, zero-centered range from −3 to +3: red means drier than the
Calibration Period, blue means wetter, and white is near normal. Values beyond ±3 remain unchanged
in `map_slices`, but their colors saturate at the scale ends; the colorbar extensions disclose this
display clipping. No drought classes are imposed. Gray cells are missing data, not near-normal
conditions.

These axes show the native latitude/longitude grid. This is not a geographic reprojection and has no
basemap, so use it for direct grid inspection rather than cartographic distance, area, or shape
interpretation.


In [ ]:
import matplotlib.pyplot as plt

plot_limit = 3.0
missing_color = "0.55"
map_cmap = plt.colormaps["RdBu"].copy()
map_cmap.set_bad(missing_color)

fig, axes = plt.subplots(ncols=2, figsize=(13, 5), constrained_layout=True)
for variable_name, index_label, ax in (
    (spi_name, "SPI", axes[0]),
    (spei_name, "SPEI", axes[1]),
):
    ax.set_facecolor(missing_color)
    map_slices[variable_name].plot(
        ax=ax,
        x="lon",
        y="lat",
        cmap=map_cmap,
        vmin=-plot_limit,
        vmax=plot_limit,
        cbar_kwargs={"label": f"{index_label} (dimensionless)", "extend": "both"},
    )
    ax.set_title(f"{index_label} | {pipeline_config['scale']}-month Timescale | {selected_date}")
    ax.set_xlabel("Longitude (degrees east)")
    ax.set_ylabel("Latitude (degrees north)")

## Time series at a selected grid cell

The maps above show one date across the grid; the plots below follow a single grid cell through the
full record, still reading from `out_ds`. Selecting a cell by coordinate uses
`.sel(lat=..., lon=..., method="nearest")`: without `method`, `.sel` demands an exact coordinate
match and raises `KeyError`, while `method="nearest"` resolves the request to the closest stored
cell. The returned value belongs to that stored cell, not to the requested coordinate, so the code
below reports both and every label uses the actual cell.

Two selections are refused rather than plotted: a coordinate outside the grid's domain, and an
all-NaN cell such as a masked ocean cell. Either would otherwise yield an empty or silently
misleading figure. A nearest grid cell is a modeled value for an area, not a station observation.


In [ ]:
def select_grid_cell(index, latitude, longitude):
    """Resolve a requested coordinate to the nearest grid cell, rejecting invalid selections."""
    lat_min, lat_max = float(index.lat.min()), float(index.lat.max())
    lon_min, lon_max = float(index.lon.min()), float(index.lon.max())
    if not (lat_min <= latitude <= lat_max and lon_min <= longitude <= lon_max):
        raise ValueError(
            f"Requested ({latitude}, {longitude}) is outside the grid domain "
            f"lat [{lat_min}, {lat_max}], lon [{lon_min}, {lon_max}]."
        )
    series = index.sel(lat=latitude, lon=longitude, method="nearest")
    actual_lat, actual_lon = float(series.lat), float(series.lon)
    if bool(series.isnull().all()):
        raise ValueError(
            f"Nearest grid cell ({actual_lat}, {actual_lon}) has no stored values (all NaN); "
            "select a cell with data."
        )
    return series, actual_lat, actual_lon


In [ ]:
# The requested coordinates sit between stored cells, so the nearest-cell
# resolution is visible instead of silently substituted.
requested_lat, requested_lon = 35.4, -99.7
spi_series, actual_lat, actual_lon = select_grid_cell(out_ds[spi_name], requested_lat, requested_lon)
spei_series, spei_lat, spei_lon = select_grid_cell(out_ds[spei_name], requested_lat, requested_lon)
assert (actual_lat, actual_lon) == (spei_lat, spei_lon)

print(f"Requested coordinate: ({requested_lat}, {requested_lon})")
print(f"Selected grid cell:   ({actual_lat:.4f}, {actual_lon:.4f})")
print(f"Timescale: {spi_series.attrs['scale']} months ({spi_series.attrs['periodicity']})")
print(f"Available values: {int(spi_series.notnull().sum())} of {spi_series.size}")


In [ ]:
# Exercise both guard paths: an out-of-domain request and an all-NaN cell
# from this grid. Neither reaches a plot.
for bad_lat, bad_lon in [(90.0, 0.0), (requested_lat, 0.0)]:
    try:
        select_grid_cell(out_ds[spi_name], bad_lat, bad_lon)
    except ValueError as exc:
        print(f"Rejected ({bad_lat}, {bad_lon}): {exc}")

# The all-NaN guard reuses the all-missing cell mask from the input
# validation above instead of scanning the full SPI output, so this
# demonstration stays at the selected-cell scale.
if bool(all_missing.any()):
    row, column = np.argwhere(all_missing.values)[0]
    nan_lat, nan_lon = float(all_missing.lat.values[row]), float(all_missing.lon.values[column])
    try:
        select_grid_cell(out_ds[spi_name], nan_lat, nan_lon)
    except ValueError as exc:
        print(f"Rejected all-NaN cell ({nan_lat:.4f}, {nan_lon:.4f}): {exc}")
else:
    print("This grid has no all-NaN cells; the all-NaN guard is covered by the test suite.")

In [ ]:
# The last stored timestep with the selected cell marked. Only that 2-D slice
# is materialized; the full grids stay on disk.
marker_time = pd.Timestamp(out_ds.time.values[-1])
marker_fig, marker_ax = plt.subplots(figsize=(7, 5))
selected_map = out_ds[spi_name].sel(time=marker_time).plot(levels=8, ax=marker_ax)
selected_map.axes.plot(
    actual_lon,
    actual_lat,
    "o",
    markersize=9,
    markerfacecolor="none",
    markeredgecolor="black",
    markeredgewidth=1.5,
    label="selected grid cell",
)
selected_map.axes.set_title(f"{spi_name} on {marker_time:%Y-%m-%d}, selected cell marked")
selected_map.axes.legend(loc="lower left", fontsize=8)


In [ ]:
import matplotlib.pyplot as plt

# The full series is materialized on purpose here: it is one cell, not the grid.
# NaNs are passed through so leading unavailable values stay a visible gap
# instead of being dropped or interpolated.
scale = spi_series.attrs["scale"]
series_fig, series_ax = plt.subplots(figsize=(12, 4.5))
series_ax.plot(
    spi_series.time.values,
    spi_series.values,
    color="tab:blue",
    linewidth=0.9,
    label=f"SPI-{scale} ({spi_series.attrs['distribution']})",
)
series_ax.plot(
    spei_series.time.values,
    spei_series.values,
    color="tab:orange",
    linewidth=0.9,
    label=f"SPEI-{scale} ({spei_series.attrs['distribution']})",
)
series_ax.axhline(0.0, color="black", linewidth=0.8, label="zero (Calibration Period normal)")
series_ax.axvspan(
    pd.Timestamp(f"{cal_start_year}-01-01"),
    pd.Timestamp(f"{cal_end_year}-12-31"),
    color="0.85",
    alpha=0.5,
    label=f"Calibration Period {cal_start_year}-{cal_end_year}",
)
series_ax.set_title(
    f"SPI-{scale} and SPEI-{scale} at grid cell ({actual_lat:.3f}, {actual_lon:.3f}) "
    f"-- {scale}-month Timescale, monthly observations"
)
series_ax.set_xlabel("Date")
series_ax.set_ylabel("Standardized index (dimensionless)")
series_ax.legend(loc="lower right", fontsize=8)
series_fig.autofmt_xdate()


### The `(time, station)` pattern is different

This walkthrough uses gridded `(time, lat, lon)` inputs, so the two spatial dimensions above are
grid axes. A station dataset stores `(time, station)`: the second dimension is a station label, and
a series is selected by that label (`.sel(station=...)`) rather than by latitude and longitude; the
statistics are otherwise the same. The distinction that matters is identity: a station series is an
observation at a point, while a grid cell is a modeled value covering an area. Selecting the
nearest cell must not be reported as selecting a station, and a station series must not be
fabricated by treating a grid cell as one.


In [ ]:
# Close the reopened handle now that the selections and plots are drawn from it.
out_ds.close()
